# 00 · Setup and Introduction to the Sage Beehive

**Course module: sensing the physical world with an edge computing network.**

Welcome. This notebook is the on-ramp for the whole series. By the end of it you
will understand what the Sage network is, how its data is organized, where the
`sage_data_client` library comes from, and you will have pulled your first real
measurement out of the network. Nothing here assumes prior experience with
sensor data or with the Sage platform.

Read the prose, not just the code. In this course the markdown cells carry the
concepts and the code cells are small enough that you can read every line. That
is deliberate: the goal is for you to understand the mechanics, not to copy an
opaque wrapper.

## 1. What Sage is

Sage is a national research network of programmable sensing computers. Each unit
in the field is a **node**: a weatherproof enclosure mounted on a light pole, a
rooftop, or a field mast, containing an on-board computer with a graphics
processor, one or more cameras, and a set of environmental sensors. A node does
not merely record numbers. It can run machine learning models locally, at the
place where the data is collected, which is what people mean by the phrase
"AI at the edge."

A few facts that set the scale and the purpose:

- Sage is a cyberinfrastructure testbed funded by the U.S. National Science
  Foundation, built to support artificial intelligence research on real world
  sensing.
- The network spans well over one hundred nodes deployed across many states,
  including fire prone regions of the western United States, urban sites, and
  ecological field stations.
- Each node carries a mix of instruments: infrared and standard cameras, air
  quality and wind sensors, and low bandwidth sensors for measurements like soil
  moisture.
- The project has operated since 2019 and grew out of, and now overlaps with,
  the earlier Chicago "Array of Things" deployment.

The lineage matters for this course because the Array of Things and the Sage
network share the same data platform, so the tools you learn here work across
both.

## 2. How the data is organized

Every node continuously produces **measurements** and ships them to a cloud
repository called the **Beehive**. You never talk to a node directly to read its
history. Instead you query the Beehive, which has already collected and stored
everything.

The single most important idea in this course is the shape of one measurement.
Every measurement is one row with the same handful of fields:

- **a timestamp**: when the reading was taken, always in UTC.
- **a name**: which quantity was measured, for example `env.temperature`.
- **a value**: the number itself.
- **metadata**: which node produced it (`meta.vsn`), which physical sensor
  (`meta.sensor`), and often where the node is (`meta.lat`, `meta.lon`).

When you run a query you get back a table of these rows, one row per reading.
Everything else in the course is manipulation of that table.

### The vocabulary you will use in every notebook

- **Node**: one physical sensing computer in the field.
- **VSN** (Virtual Sensor Name): the node's short call sign, for example `W06C`
  or `N001`. This is the identifier you filter queries on. The examples in these
  notebooks use real call signs, but which nodes are online changes over time, so
  notebook `01` teaches you how to discover the nodes that are live right now.
- **Metric**: a named measurement stream. Names are dotted, grouping a family and
  a specific quantity: `env.temperature`, `env.relative_humidity`, `sys.uptime`.
- **Beehive**: the cloud service that stores every measurement and answers your
  queries over HTTP.
- **Plugin**: a program that runs on a node and produces measurements or
  processes camera frames. You do not need plugins for this course, but you will
  see the word `plugin` appear as a filter key.

## 3. Where `sage_data_client` comes from

You will type `sage_data_client` in nearly every cell of this course, so it is
worth knowing exactly what it is.

- It is the **official** Sage data client, maintained by the Sage project itself
  in the repository `github.com/sagecontinuum/sage-data-client`. It is a
  first party library, not something written for this course.
- On the Python Package Index its name is `sage-data-client` (with hyphens). The
  name you import in code is `sage_data_client` (with underscores). This hyphen
  versus underscore split is a normal Python convention: you install with one
  spelling and import with the other.
- The library is small on purpose. It exposes exactly two functions:
  - `query(...)`: fetch measurements from the Beehive into a pandas DataFrame.
  - `load(...)`: read a previously saved query result back from a file or URL.
- Under the hood `query` sends an HTTP request to
  `https://data.sagecontinuum.org/api/v1/query` and hands you back the result as
  a table.

Two libraries, do not confuse them:

| Library | What it is | Where it lives |
| --- | --- | --- |
| `sage_data_client` | official thin HTTP client for the Beehive | `sagecontinuum/sage-data-client` |
| `SageTools` (`sage`) | convenience wrappers built on top of it | `mpapka/SageTools` |

This course teaches the official client first, then points to the SageTools
wrapper that packages each pattern.

## 4. Install the tools

Run the cell below once. It installs four things:

- `sage-data-client`, the Beehive client described above.
- `pandas`, the table library that every query result is expressed in.
- `matplotlib`, used for plotting starting in notebook `05`.
- `numpy`, used for numeric work and by the synthetic data generators in the
  offline notebooks.

On a managed classroom JupyterHub these are often preinstalled, in which case the
cell finishes almost instantly. The `--quiet` flag keeps the output short.

In [ ]:
%pip install --quiet sage-data-client pandas matplotlib numpy
print("install step complete")

### Optional: install the SageTools helper package

This is optional. Every lab also works without it. To turn on the SageTools
convenience functions, uncomment the `%pip` line below and run the cell.

Two things make this cell behave in a notebook, where a naive install often does
not:

- It uses the `%pip` magic, not a shell `!pip`. `%pip` installs into the exact
  Python environment backing this kernel, so the package lands where your later
  `import sage` will look for it. Plain command line pip does not guarantee that.
- It installs straight from GitHub as a normal, non-editable install. Because the
  files go into site-packages, which is already on the import path, the import
  cell right below picks them up with no kernel restart.

Avoid `pip install -e` (editable mode) here. An editable install reports success
but is not importable until you restart the kernel, because Python only reads the
editable path entry at interpreter startup. In a live class that looks like a
broken import even though pip said it worked.

If you would rather have the source tree on disk to read or edit, use the
commented `sys.path` approach instead. It clones the repo, puts it on the path
immediately with no install and no restart, and is safe to re-run.

In [ ]:
# OPTIONAL. Uncomment the line below to enable the SageTools helpers, then run.
# %pip install --quiet "git+https://github.com/mpapka/SageTools.git"

# Alternative, if you want the source tree on disk. This clones the repo and puts
# it on the import path immediately, with no install and no kernel restart:
#
#   import os, sys, subprocess
#   repoDir = "SageTools"
#   if not os.path.isdir(repoDir):
#       subprocess.run(["git", "clone", "https://github.com/mpapka/SageTools.git"], check=True)
#   if repoDir not in sys.path:
#       sys.path.insert(0, repoDir)

print("Optional. Uncomment one approach above to enable the SageTools helpers.")

In [ ]:
# Import the SageTools helper package (github.com/mpapka/SageTools).
#
# SageTools is a convenience layer built on top of the official sage_data_client.
# It is optional. Every lab in this course also shows the underlying
# sage_data_client call directly, so the helper is never a hard dependency.
#
# The try/except means the notebook runs whether or not the package is installed:
# if it is missing, haveSageTools becomes False and the notebook falls back to
# the direct calls it already teaches.
try:
    import sage
    haveSageTools = True
except ImportError:
    haveSageTools = False

print("SageTools helper package available:", haveSageTools)

## 5. First contact: one real query

The cell below is the whole course in miniature. It calls `query` and asks the
Beehive a very small question: "give me the single most recent temperature
reading from anywhere in the network in the last five minutes."

Read the arguments before you run it:

- `start="-5m"`: begin the search window five minutes ago. The minus sign means
  "relative to now." You will see the full grammar of time arguments in notebook
  `02`.
- `filter={"name": "env.temperature"}`: only return temperature readings. Without
  this filter you would get every metric on every node, which is a lot.
- `tail=1`: of everything that matches, return only the last (most recent) row.
  Asking for one row keeps the query fast and cheap.

This is a **live** call. It reaches out over the internet to the real Beehive.
There is no synthetic or fake data anywhere in this notebook. If it returns rows,
those rows are genuine readings from physical sensors. If you are offline it will
raise an error, which is expected. The analysis and plotting notebooks (`04`,
`05`, `06`) carry offline data so you can still work without a network, but this
one does not.

In [ ]:
import sage_data_client
import pandas as pd

probeDf = sage_data_client.query(
    start="-5m",
    filter={"name": "env.temperature"},
    tail=1,
)

print("rows returned:", len(probeDf))
probeDf.head()

## 6. Reading one measurement record, field by field

A table is only useful if you know what each column means. The cell below takes
the single row you just fetched and prints every field with an explanation. Run
it after the query above.

The important columns:

- `timestamp`: when the reading was taken, in UTC. This is your proof of
  liveness: a reading that is only minutes old cannot be pre-generated.
- `name`: the metric, here `env.temperature`.
- `value`: the measured number. For `env.temperature` the unit is **degrees
  Celsius**. Different metrics carry different units, so always confirm what a
  metric means before you interpret its number.
- `meta.vsn`: the call sign of the node that produced the reading.
- `meta.sensor`: the specific physical sensor on that node, since a node can
  carry more than one temperature sensor.

In [ ]:
from datetime import datetime, timezone

if probeDf.empty:
    print("No rows came back. The query reached the Beehive but found no")
    print("temperature reading in the last 5 minutes. Widen the window: change")
    print("start to '-1h' in the cell above and run it again.")
else:
    latestRow = probeDf.iloc[0]

    readingTime = pd.to_datetime(latestRow["timestamp"])
    ageMinutes = (datetime.now(timezone.utc) - readingTime).total_seconds() / 60
    nodeVsn = latestRow["meta.vsn"]

    print("timestamp   :", readingTime, "UTC")
    print("metric name :", latestRow["name"])
    print("value       :", round(float(latestRow["value"]), 2), "degrees Celsius")
    print("node (vsn)  :", nodeVsn)
    print("sensor      :", latestRow.get("meta.sensor", "not reported"))
    print("reading age :", round(ageMinutes, 1), "minutes old")
    print()

    if ageMinutes < 10:
        print("This is live data. A reading minutes old cannot be synthetic.")
    print("Look this node up in the Sage portal to see the physical unit:")
    print(f"  https://portal.sagecontinuum.org/nodes/{nodeVsn}")

## 7. Troubleshooting your first query

A short field guide to what can go wrong, so you can tell a real problem from a
non problem.

- **`rows returned: 0` and an empty table.** The query worked, it just found
  nothing in the window. Not every metric reports every five minutes. Widen the
  window (`start="-1h"`) or drop the `name` filter to see any recent traffic.
- **A network or connection error.** Your environment cannot reach
  `data.sagecontinuum.org`. On a locked down campus or corporate network that
  host may need to be allowed through the firewall. This is a reachability issue,
  not a data issue.
- **`ModuleNotFoundError: sage_data_client`.** The install cell in section 4 did
  not run, or ran in a different environment than this kernel. Re-run it, then
  re-run the import.
- **A value that looks implausible**, for example a temperature of 85 when you
  expected 20. Check the unit. The Beehive reports temperature in Celsius. An 85
  Celsius reading from an outdoor sensor is a hardware fault worth noting, but an
  85 in a column you assumed was Fahrenheit is just a units mix up.

## 8. What you can do now, and where to go next

You have the full loop in miniature: install the client, call `query` with a
time window and a filter, and read the resulting table. Everything in the rest of
the course expands one piece of that loop.

- Notebook `01` Discovery: how to find which nodes are live and what each one
  measures, so you are not guessing at call signs.
- Notebook `02` Querying: the complete grammar of time windows and filters, and a
  full tour of the returned DataFrame.
- Notebook `03` Scaling up: pulling months of data without timeouts.
- Notebooks `04` and `05`: analysis and visualization, which run offline.
- Notebook `06` Capstone: an end to end regional study.

### Exercises

1. Re-run the probe query with `start="-1h"` and `tail=5`. How many rows come
   back now, and are they all from the same node or from several?
2. Change the filter to `{"name": "env.relative_humidity"}` and read the value.
   What unit do you think humidity is reported in? State your reasoning.
3. Take the `meta.vsn` from your result and open its portal page using the URL
   printed in section 6. What kind of site is that node deployed at?
4. Explain in one sentence, to someone who has not taken this course, the
   difference between `sage_data_client` and `SageTools`.

## Further reading

**Sage**

- Sage project home: https://sagecontinuum.org/
- Sage documentation: https://sagecontinuum.org/docs/reference-guides/dev-quick-reference
- Sage portal, browse live nodes: https://portal.sagecontinuum.org/
- Sage cyberinfrastructure source: https://github.com/sagecontinuum/sage
- The official data client, `sage-data-client`: https://github.com/sagecontinuum/sage-data-client and https://pypi.org/project/sage-data-client/
- SageTools, the helper layer used in this course: https://github.com/mpapka/SageTools

**Technology**

- pandas, the table library every result uses: https://pandas.pydata.org/docs/
- Jupyter documentation: https://docs.jupyter.org/